In [ ]:
!pip install -q gradio scikit-learn==1.9.1

import os, sys, importlib, shutil
import joblib, numpy as np, torch
from PIL import Image
from torchvision import transforms
from transformers import AutoModel

# get the project files from GitHub
REPO_URL = "https://github.com/zurimckee/plantmodeltraining.git"
FOLDER   = "plant_health"        # folder in the repo holding the 5 files
shutil.rmtree("/content/repo", ignore_errors=True)
!git clone -q {REPO_URL} /content/repo

DATA_DIR = f"/content/repo/{FOLDER}"
PROBE    = f"{DATA_DIR}/species_dinov2_base_probe.joblib"

# check that every file is where it should be
for p in [PROBE, f"{DATA_DIR}/plant_gui.py", f"{DATA_DIR}/advice.csv",
          f"{DATA_DIR}/species_groups.csv", f"{DATA_DIR}/symptoms.csv"]:
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)

device = "cuda" if torch.cuda.is_available() else "cpu"
EVAL_TF = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class SpeciesPredictor:
    def __init__(self, path, device):
        d = joblib.load(path)
        self.scaler, self.clf, self.classes = d["scaler"], d["clf"], d["classes"]
        self.device = device
        self.backbone = AutoModel.from_pretrained(d["backbone"]).eval().to(device)

    @torch.no_grad()
    def predict(self, img, k=3):
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        x = EVAL_TF(img.convert("RGB")).unsqueeze(0).to(self.device)
        f = self.backbone(pixel_values=x).pooler_output.float().cpu().numpy()
        p = self.clf.predict_proba(self.scaler.transform(f))[0]
        top = np.argsort(-p)[:k]
        return {"top_k": [(self.classes[i].replace("_", " "), float(p[i])) for i in top]}

predictor = SpeciesPredictor(PROBE, device)
predict = lambda img, model, classes, k=3: predictor.predict(img, k)

sys.path.append(DATA_DIR)
import plant_gui
importlib.reload(plant_gui)
plant_gui.build_demo(predictor.backbone, predictor.classes, predict, DATA_DIR).launch(share=True, debug=True)

OK      /content/repo/plant_health/species_dinov2_base_probe.joblib
OK      /content/repo/plant_health/plant_gui.py
OK      /content/repo/plant_health/advice.csv
OK      /content/repo/plant_health/species_groups.csv
OK      /content/repo/plant_health/symptoms.csv


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

NOTE no advice rows yet for: ['Dandelion']


/content/repo/plant_health/plant_gui.py:293: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, title="Plant Identifier") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://74dbd73c78eca90566.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://74dbd73c78eca90566.gradio.live
